In [ ]:
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, precision_recall_fscore_support

# =========================
# LOAD DATA
# =========================
df = pd.read_csv("dataset/machine_failure.csv")
df = df.drop(columns=["UDI", "Product ID", "Type"])

feature_cols = [c for c in df.columns if c not in
                ["Machine failure", "TWF", "HDF", "PWF", "OSF", "RNF"]]

X = df[feature_cols]
targets = ["TWF", "HDF", "PWF", "OSF", "RNF"]

# =========================
# TRAIN / TEST SPLIT
# =========================
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)

# =========================
# SCALER + LOF (NORMAL ONLY)
# =========================
normal_df = df[df["Machine failure"] == 0]
X_normal = normal_df[feature_cols]

scaler = StandardScaler()
X_normal_scaled = scaler.fit_transform(X_normal)

lof = LocalOutlierFactor(
    n_neighbors=35,
    contamination=0.05,
    novelty=True
)
lof.fit(X_normal_scaled)

def anomaly_score(model, X):
    raw = model.decision_function(X)
    return 1 / (1 + np.exp(-raw))

# =========================
# SCALE DATA
# =========================
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {}

# =========================
# TRAIN MODELS + METRICS
# =========================
print("\n=========================")
print("TRAINING MODELS")
print("=========================")

for target in targets:
    print(f"\n========== {target} ==========")

    y_train = df.loc[X_train.index, target]
    y_test = df.loc[X_test.index, target]

    smote = SMOTE(random_state=42)
    X_res, y_res = smote.fit_resample(X_train_scaled, y_train)

    model = RandomForestClassifier(
        n_estimators=500,
        max_depth=14,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_res, y_res)
    models[target] = model

    # =========================
    # TRAIN METRICS
    # =========================
    train_pred = model.predict(X_res)
    print("\n--- TRAIN METRICS ---")
    print(classification_report(y_res, train_pred, zero_division=0))

    # =========================
    # TEST METRICS
    # =========================
    test_pred = model.predict(X_test_scaled)
    print("\n--- TEST METRICS ---")
    print(classification_report(y_test, test_pred, zero_division=0))

# =========================
# MULTI-LABEL TEST METRICS
# =========================
print("\n=========================")
print("OVERALL MULTI-LABEL METRICS")
print("=========================")

y_true_all = []
y_pred_all = []

for t in targets:
    y_true_all.append(df.loc[X_test.index, t].values)
    y_pred_all.append(models[t].predict(X_test_scaled))

y_true_all = np.array(y_true_all).T
y_pred_all = np.array(y_pred_all).T

p, r, f1, _ = precision_recall_fscore_support(
    y_true_all, y_pred_all, average="micro"
)

print("Micro Precision:", p)
print("Micro Recall:", r)
print("Micro F1:", f1)

# =========================
# CUSTOM LOF + FAULT METRICS
# =========================
print("\n=========================")
print("CUSTOM SYSTEM METRICS")
print("=========================")

THRESHOLD = np.percentile(anomaly_score(lof, X_test_scaled), 90)

TP = TN = FP = FN = 0

for i in range(len(X_test_scaled)):
    score = anomaly_score(lof, X_test_scaled)[i]
    has_fault = np.any(y_true_all[i] == 1)
    predicted_fault = np.any(y_pred_all[i] == 1)

    if score > THRESHOLD and predicted_fault:
        TP += 1
    elif score <= THRESHOLD and not predicted_fault:
        TN += 1
    elif score > THRESHOLD and not predicted_fault:
        FP += 1
    elif score <= THRESHOLD and predicted_fault:
        FN += 1

precision = TP / (TP + FP + 1e-9)
recall = TP / (TP + FN + 1e-9)
f1 = 2 * precision * recall / (precision + recall + 1e-9)

print(f"TP: {TP}")
print(f"TN: {TN}")
print(f"FP: {FP}")
print(f"FN: {FN}")

print("\nPrecision:", precision)
print("Recall:", recall)
print("F1:", f1)

# =========================
# DECISION LOGIC
# =========================
def final_decision(score, fv):
    if score > 0.7:
        return "FAILURE"
    elif score > 0.4 or sum(fv) > 0:
        return "WARNING"
    else:
        return "NORMAL"

# =========================
# PREDICT PIPELINE
# =========================
def predict(X_input):
    X_scaled = scaler.transform(X_input)

    score = anomaly_score(lof, X_scaled)[0]

    fv = [int(models[t].predict(X_scaled)[0]) for t in targets]

    return {
        "anomaly_score": float(score),
        "failure_vector": fv,
        "decision": final_decision(score, fv)
    }

# =========================
# SAMPLE TESTS
# =========================
sample = pd.DataFrame([{
    "Air temperature [K]": 298.2,
    "Process temperature [K]": 308.5,
    "Rotational speed [rpm]": 1400,
    "Torque [Nm]": 65.0,
    "Tool wear [min]": 191
}])

sample2 = pd.DataFrame([{
    "Air temperature [K]": 298.2,
    "Process temperature [K]": 308.5,
    "Rotational speed [rpm]": 1400,
    "Torque [Nm]": 30,
    "Tool wear [min]": 106
}])

print("\n=========================")
print("SAMPLE OUTPUTS")
print("=========================")

print(predict(sample))
print(predict(sample2))

# =========================
# SAVE MODELS
# =========================
joblib.dump(lof, "lof_model.joblib")
joblib.dump(models, "rf_models.joblib")
joblib.dump(scaler, "scaler.joblib")
joblib.dump(feature_cols, "feature_cols.joblib")

print("\nModels saved successfully!")


TRAINING MODELS

========== TWF ==========

--- TRAIN METRICS ---
              precision    recall  f1-score   support

           0       1.00      0.97      0.99      7965
           1       0.97      1.00      0.99      7965

    accuracy                           0.99     15930
   macro avg       0.99      0.99      0.99     15930
weighted avg       0.99      0.99      0.99     15930


--- TEST METRICS ---
              precision    recall  f1-score   support

           0       1.00      0.97      0.98      1989
           1       0.09      0.55      0.15        11

    accuracy                           0.97      2000
   macro avg       0.54      0.76      0.57      2000
weighted avg       0.99      0.97      0.98      2000


========== HDF ==========

--- TRAIN METRICS ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      7902
           1       1.00      1.00      1.00      7902

    accuracy                           1.00